In [1]:
import pandas as pd
import pyodbc

In [2]:
server = 'ITT-GARIMA-P\\SQLEXPRESS'
database = 'SalesData'
table = 'sales_data'

conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'

conn = None
try:
    conn = pyodbc.connect(conn_str)
    print("Connection to SQL Server successful!")
except Exception as e:
    print(f"Error: Could not connect to SQL Server. Please check your connection details.\n{e}")

Connection to SQL Server successful!


In [3]:
df = None
if conn:
    try:
        query = f"SELECT * FROM {table}"
        df = pd.read_sql(query, conn)
        print(f"Data fetched successfully from table '{table}'!")
        conn.close()
        print("Connection closed.")
    except Exception as e:
        print(f"Error fetching data: {e}")
else:
    print("Cannot fetch data, connection not established.")

C:\Users\garima.parmar\AppData\Local\Temp\ipykernel_31688\1751968590.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data fetched successfully from table 'sales_data'!
Connection closed.


In [4]:
display(df.head())

,OrderID,Product,Quantity,Price,OrderDate,Address
0,192873,Google Phone,1.0,1212.07,04/12/19 15:23,"563 Oak St, Miami, FL 38994"
1,152014,Apple Airpods,1.0,526.04,10/25/19 19:50,"257 Oak St, Los Angeles, CA 57733"
2,145859,AA Batteries (4-pack),1.0,1169.5,03/16/19 09:51,"522 3rd St, Denver, CO 61868"
3,124079,Wired Headphones,1.0,11.4,04/12/19 17:29,"304 1st St, Atlanta, GA 50231"
4,141228,Lightning Charging Cable,1.0,1146.91,12/26/19 19:17,"871 Main St, New York, NY 71395"


In [5]:
def get_category(product_name):
    if not isinstance(product_name, str):
        return 'Other'
    product_name = product_name.lower()
    if 'iphone' in product_name or 'google phone' in product_name or 'vareebadd phone' in product_name:
        return 'Phone'
    elif 'headphones' in product_name or 'airpods' in product_name:
        return 'Audio'
    elif 'cable' in product_name or 'charger' in product_name or 'batteries' in product_name:
        return 'Accessory'
    elif 'laptop' in product_name or 'monitor' in product_name:
        return 'Computer & Display'
    else:
        return 'Other'

def get_zipcode(address):
    try:
        return address.split(',')[-1].strip().split(' ')[-1]
    except (IndexError, AttributeError):
        return None


In [6]:
def remove_duplicates(df):
    return df.drop_duplicates()

def handle_missing(df, required_cols):
    return df.dropna(subset=required_cols)

def convert_types(df, quantity_col, price_col, date_col):
    df[quantity_col] = pd.to_numeric(df[quantity_col], errors='coerce')
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    return df

def enrich_time_data(df, date_col):
    df['Hour'] = df[date_col].dt.hour
    df['DayOfWeek'] = df[date_col].dt.day_name()
    df['Year'] = df[date_col].dt.year
    df['Month'] = df[date_col].dt.month
    return df

def calculate_sales(df, quantity_col, price_col):
    df['Sales'] = df[quantity_col] * df[price_col]
    return df

def extract_location_details(df, address_col):
    df['City'] = df[address_col].apply(lambda x: x.split(',')[1].strip() if pd.notnull(x) and len(x.split(',')) > 1 else None)
    df['State'] = df[address_col].apply(lambda x: x.split(',')[2].split()[0] if pd.notnull(x) and len(x.split(',')) > 2 else None)
    df['ZipCode'] = df[address_col].apply(get_zipcode)
    return df

def categorize_products(df):
    df['Category'] = df['Product'].apply(get_category)
    return df


In [7]:
def clean_data(df):
    df = df.copy()

    quantity_col = 'Quantity'
    price_col = 'Price'
    date_col = 'OrderDate'
    address_col = 'Address'

    df = remove_duplicates(df)
    df = handle_missing(df, [quantity_col, price_col, date_col])
    df = convert_types(df, quantity_col, price_col, date_col)
    df = enrich_time_data(df, date_col)
    df = calculate_sales(df, quantity_col, price_col)
    df = extract_location_details(df, address_col)
    df = categorize_products(df)

    print("Data cleaning and transformations complete.")
    return df


In [8]:
if df is not None:
    df_cleaned = clean_data(df)
else:
    print("DataFrame is empty. Cannot perform transformations.")


C:\Users\garima.parmar\AppData\Local\Temp\ipykernel_31688\1635188899.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


Data cleaning and transformations complete.


In [9]:
df_cleaned['OrderID'] = df_cleaned['OrderID'].astype(str)
df_cleaned['Product'] = df_cleaned['Product'].astype(str)
df_cleaned['Quantity'] = pd.to_numeric(df_cleaned['Quantity'], errors='coerce').astype('Int64')
df_cleaned['Price'] = pd.to_numeric(df_cleaned['Price'], errors='coerce')
df_cleaned['OrderDate'] = pd.to_datetime(df_cleaned['OrderDate'], errors='coerce')
df_cleaned['Address'] = df_cleaned['Address'].astype(str)
df_cleaned['Hour'] = pd.to_numeric(df_cleaned['Hour'], errors='coerce').astype('Int64')
df_cleaned['Sales'] = pd.to_numeric(df_cleaned['Sales'], errors='coerce')
df_cleaned['DayOfWeek'] = df_cleaned['DayOfWeek'].astype(str)
df_cleaned['Year'] = pd.to_numeric(df_cleaned['Year'], errors='coerce').astype('Int64')
df_cleaned['ZipCode'] = df_cleaned['ZipCode'].astype(str)
df_cleaned['Category'] = df_cleaned['Category'].astype(str)
df_cleaned['Month'] = pd.to_numeric(df_cleaned['Month'], errors='coerce').astype('Int64')
df_cleaned['City'] = df_cleaned['City'].astype(str)
df_cleaned['State'] = df_cleaned['State'].astype(str)

In [10]:
df_cleaned = df_cleaned.sort_values('OrderDate')
df_cleaned['CumulativeSales'] = df_cleaned['Sales'].cumsum()

In [11]:
df_cleaned['Rolling7DaySales'] = df_cleaned['Sales'].rolling(window=7, min_periods=1).sum()

In [12]:
region_map = {
    'CA': 'West', 'NY': 'Northeast', 'FL': 'South', 'TX': 'South'
}
df_cleaned['Region'] = df_cleaned['State'].map(region_map)

In [13]:
bins = [0, 1000, 5000, 10000, df_cleaned['Sales'].max()]
labels = ['Low', 'Medium', 'High', 'Very High']
df_cleaned['OrderValueCategory'] = pd.cut(df_cleaned['Sales'], bins=bins, labels=labels)

In [14]:
order_counts = df_cleaned.groupby('OrderID')['Product'].count()
df_cleaned['IsBundle'] = df_cleaned['OrderID'].map(lambda x: order_counts[x] > 1)

In [15]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
df_cleaned['Season'] = df_cleaned['Month'].apply(get_season)

In [16]:
display(df_cleaned.head())

,OrderID,Product,Quantity,Price,OrderDate,Address,Hour,DayOfWeek,Year,Month,...,City,State,ZipCode,Category,CumulativeSales,Rolling7DaySales,Region,OrderValueCategory,IsBundle,Season
15480,133233,AA Batteries (4-pack),2,1507.97,2019-01-01 00:00:00,"561 3rd St, Miami, FL 30592",0,Tuesday,2019,1,...,Miami,FL,30592,Accessory,3015.94,3015.94,South,Medium,False,Winter
89955,124783,LG Washing Machine,1,715.66,2019-01-01 00:00:00,"142 2nd St, San Francisco, CA 23808",0,Tuesday,2019,1,...,San Francisco,CA,23808,Other,3731.60,3731.60,West,Low,False,Winter
86426,164666,34in Ultrawide Monitor,1,663.46,2019-01-01 00:03:00,"344 Oak St, Dallas, TX 66555",0,Tuesday,2019,1,...,Dallas,TX,66555,Computer & Display,4395.06,4395.06,South,Low,False,Winter
56918,182095,27in FHD Monitor,1,93.70,2019-01-01 00:05:00,"584 2nd St, Denver, CO 66960",0,Tuesday,2019,1,...,Denver,CO,66960,Computer & Display,4488.76,4488.76,NaN,Low,False,Winter
88227,171725,34in Ultrawide Monitor,1,330.79,2019-01-01 00:16:00,"721 2nd St, Miami, FL 30890",0,Tuesday,2019,1,...,Miami,FL,30890,Computer & Display,4819.55,4819.55,South,Low,False,Winter


Data Visualizations

In [17]:
import plotly.express as px
import calendar

monthly_sales_by_year = df_cleaned.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly_sales_by_year = monthly_sales_by_year.sort_values(['Year', 'Month'])
monthly_sales_by_year['MonthName'] = monthly_sales_by_year['Month'].apply(lambda x: calendar.month_abbr[int(x)])

monthly_sales_by_year['Year'] = monthly_sales_by_year['Year'].astype(str)

fig = px.line(monthly_sales_by_year, 
              x='MonthName', 
              y='Sales', 
              color='Year',
              title='Total Sales per Month by Year',
              labels={'MonthName': 'Month', 'Sales': 'Total Sales ($)', 'Year': 'Year'},
              markers=True)

fig.update_traces(hovertemplate='<b>Month:</b> %{x}<br><b>Sales:</b> $%{y:,.2f}')
fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Total Sales ($)',
    title_x=0.5,
    xaxis={'categoryorder':'array', 'categoryarray': [calendar.month_abbr[i] for i in range(1, 13)]}
)
fig.show()

In [18]:
import plotly.graph_objects as go
import calendar

monthly_category_sales = df_cleaned.groupby(['Month', 'Category'])['Sales'].sum().reset_index()

months = sorted(monthly_category_sales['Month'].unique())
month_names = [calendar.month_abbr[int(m)] for m in months]

fig = go.Figure()

for month in months:
    df_month = monthly_category_sales[monthly_category_sales['Month'] == month]
    fig.add_trace(
        go.Bar(
            x=df_month['Category'],
            y=df_month['Sales'],
            name=calendar.month_abbr[int(month)],
            visible=False
        )
    )

total_category_sales = df_cleaned.groupby('Category')['Sales'].sum().reset_index()
fig.add_trace(
    go.Bar(
        x=total_category_sales['Category'],
        y=total_category_sales['Sales'],
        name='All Months',
        visible=True
    )
)

fig.update_traces(
    texttemplate='$%{y:,.2s}', 
    textposition='inside',
    textfont=dict(
        color='white',
        size=12
    ),
    insidetextanchor='end',
    selector=dict(type='bar')
)

buttons = []

buttons.append(dict(
    label="All Months",
    method="update",
    args=[{"visible": [False]*len(months) + [True]},
          {"title": "Total Sales by Product Category (All Months)"}]
))

for i, month_name in enumerate(month_names):
    visibility = [False] * (len(months) + 1)
    visibility[i] = True
    buttons.append(dict(
        label=month_name,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Total Sales by Category for {month_name}"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    title_text="Total Sales by Product Category",
    xaxis_title="Product Category",
    yaxis_title="Total Sales ($)",
    title_x=0.5
)

fig.show()

In [19]:
top_products = df_cleaned.groupby('Product')['Quantity'].sum().nlargest(10).reset_index()

fig = px.bar(top_products, 
             x='Quantity', 
             y='Product', 
             orientation='h',
             title='Top 10 Selling Products by Quantity',
             labels={'Quantity': 'Total Quantity Sold', 'Product': 'Product'})
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [20]:
city_sales = df_cleaned[df_cleaned['City'] != 'None'].groupby('City')['Sales'].sum().nlargest(10).reset_index()

fig = px.bar(city_sales, 
             x='Sales', 
             y='City', 
             orientation='h',
             title='Top 10 Cities by Sales',
             labels={'Sales': 'Total Sales ($)', 'City': 'City'},
             text='Sales')
fig.update_traces(texttemplate='$%{text:,.2s}')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [21]:
hourly_orders = df_cleaned.groupby('Hour')['OrderID'].count().reset_index()

fig = px.line(hourly_orders, 
              x='Hour', 
              y='OrderID', 
              title='Number of Orders per Hour',
              labels={'Hour': 'Hour of Day', 'OrderID': 'Number of Orders'},
              markers=True)
fig.update_xaxes(dtick=1)
fig.show()

In [22]:
fig = px.treemap(df_cleaned, 
                 path=[px.Constant("All Sales"), 'Category', 'Product'], 
                 values='Sales',
                 title='Sales Treemap by Category and Product',
                 hover_data=['Quantity'])
fig.update_traces(textinfo="label+percent entry")
fig.show()